<a href="https://colab.research.google.com/github/johanmorpar/Data_Science_Projects/blob/main/RapiCredit_PruebaRiesgo_DataScientist_30sep26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prueba Técnica Data Scientist - RapiCredit (Johan Moreno)
## Modelo de riesgo de mora

**Cómo está organizado este notebook**
1. Cargar las tres bases y relacion de datos.
2. Analisis exploratorio de datos.
3. Hallazgos sobre la base de datos.
4. Preparación de los datos y partición entrenamiento (80%) y prueba (20%).
5. Modelado.
6. Resultados, punto de corte y segmentos.
7. Escenarios extra.
8. Conclusiones.


**Clave:** mora = (`Is_1Plus_DPD = 1`). Aprobar a alguien que luego no paga es un **falso negativo**. Rechazar a alguien que sí habría pagado es un **falso positivo**.

---
# Sección 1. Cargar las tres bases y relacionarlas

Tenemos tres archivos CSV que describen el mismo crédito desde tres ángulos para generar un solo dataframe.

| Base | Qué tiene | Llave |
|---|---|---|
| Clients | Datos del cliente | `Client_ID` |
| Loans | Datos de originación del crédito | `Loan_ID` y `Client_ID` |
| Performance | Comportamiento posterior y target | `Loan_ID` |


La relación es: **Clients 1 a N Loans** (un cliente puede tener varios préstamos) y **Loans 1 a 1 Performance** (cada préstamo tiene un solo registro de comportamiento).

### 1.1 Cargar los archivos

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

RUTA = "/content/drive/MyDrive/RapiCredit"   # carpeta donde están los CSV

clients = pd.read_csv(f"{RUTA}/Clients_Clean.csv")
loans   = pd.read_csv(f"{RUTA}/Loans_Clean.csv", sep=";")   # Loans viene separado por punto y coma
perf    = pd.read_csv(f"{RUTA}/Performance_Clean.csv")

for nombre, tabla in [("Clients", clients), ("Loans", loans), ("Performance", perf)]:
    print(f"{nombre:12s} filas: {tabla.shape[0]:>6,}   columnas: {tabla.shape[1]}")

Mounted at /content/drive
Clients      filas: 10,000   columnas: 8
Loans        filas: 10,000   columnas: 10
Performance  filas: 10,000   columnas: 6


### 1.2 Verificar las llaves antes de unir
todos deben dar 0. Si una llave estuviera repetida, al unir se duplicarían filas y todas las métricas quedarían infladas sin avisar.

In [3]:
print("Client_ID repetido en Clients:     ", clients["Client_ID"].duplicated().sum())
print("Loan_ID repetido en Loans:         ", loans["Loan_ID"].duplicated().sum())
print("Loan_ID repetido en Performance:   ", perf["Loan_ID"].duplicated().sum())
print("Préstamos sin cliente en Clients:  ", (~loans["Client_ID"].isin(clients["Client_ID"])).sum())
print("Préstamos sin registro Performance:", (~loans["Loan_ID"].isin(perf["Loan_ID"])).sum())

Client_ID repetido en Clients:      0
Loan_ID repetido en Loans:          0
Loan_ID repetido en Performance:    0
Préstamos sin cliente en Clients:   0
Préstamos sin registro Performance: 0


### 1.3 Unir las tres bases

In [4]:
df = (
    loans
    .merge(clients, on="Client_ID", how="left", validate="m:1")   # muchos préstamos, un cliente
    .merge(perf,    on="Loan_ID",   how="left", validate="1:1")   # un registro de comportamiento por préstamo
)

# Las fechas vienen como texto día/mes/año: se convierten a fecha
for col in ["Fecha_Desembolso", "Fecha_Vencimiento"]:
    df[col] = pd.to_datetime(df[col], format="%d/%m/%Y")

# Control: la unión no debe cambiar el número de filas
assert len(df) == len(loans), "La unión cambió el número de filas"
print("Base unida:", df.shape[0], "filas y", df.shape[1], "columnas")

Base unida: 10000 filas y 22 columnas


- Loans (originacion) es la base central porque tiene el análisis de cada préstamo.
- **`how="left"`:** conserva todos los registros de credito aunque falte algo en la otra tabla. Si hubiera faltantes, se verían como null en lugar de desaparecer.
- **`validate`** le dice a pandas qué relación esperamos. Si la suposición es falsa, frena el código en lugar de unir mal.
- **`assert`** confirma que no se perdieron ni se duplicaron filas

# Sección 2. Inspección de la base unida

**Con la base lista, lseguimos al analisis exporatorio de la nueva base: tamaño, tipos, nulos, duplicados, estadísticas, categorías y target. Cada paso responde una pregunta distinta.**

### 2.1 Tamaño (.shape) y primeras filas (.head)

In [5]:
print(df.shape)
df.head(3)

(10000, 22)


,Loan_ID,Client_ID,Fecha_Desembolso,Fecha_Vencimiento,Loan_Amount,Term_Days,Dispersion_Method,Loan_Purpose,Historical_Loan_Count,Estado_Credito,Tipo_Cliente,Monthly_Income,Age,City_Id,Region,Employment_Type,Internal_Score,Total_Due,Current_Loan_Novations,Rango_NPL,Days_Since_First_Loan,Is_1Plus_DPD
0,6037124,500000,2025-12-31,2026-01-30,500000,30,Cuenta_Bancaria,Otros,7,PAGADO,Recurrente,5860000,62,9,BOLIVAR,Empleado,467.0,576609,5,02. Timely Payment,1651.0,0
1,5933585,500001,2025-12-13,2026-01-12,320000,30,Cuenta_Bancaria,Otros,5,PAGADO,Recurrente,4559000,38,645,CUNDINAMARCA,Empleado,NaN,393333,1,03. Untimely Payment (1+ DPD),1267.0,1
2,6588760,500002,2026-04-08,2026-05-08,250000,30,Cuenta_Bancaria,Otros,1,PAGADO,Recurrente,3200000,34,1,ANTIOQUIA,Independiente,NaN,345093,0,03. Untimely Payment (1+ DPD),82.0,1


### 2.2 Tipos de datos (.dtypes)


In [6]:
df.dtypes

,0
Loan_ID,int64
Client_ID,int64
Fecha_Desembolso,datetime64[ns]
Fecha_Vencimiento,datetime64[ns]
Loan_Amount,int64
Term_Days,int64
Dispersion_Method,object
Loan_Purpose,object
Historical_Loan_Count,int64
Estado_Credito,object


### 2.3 Nulos (.df.isna)

In [7]:
nulos = df.isna().sum().to_frame("nulos")
nulos["% del total"] = (nulos["nulos"] / len(df) * 100).round(1)
nulos[nulos["nulos"] > 0]

,nulos,% del total
Internal_Score,8163,81.6
Days_Since_First_Loan,1213,12.1


solo dos columnas tienen nulos: `Internal_Score` (8.163, el **81,6%**) y `Days_Since_First_Loan` (1.213, el **12,1%**).

### 2.4 Duplicados

In [8]:
print("Filas completamente duplicadas:", df.duplicated().sum())
print("Loan_ID únicos:", df["Loan_ID"].is_unique)

Filas completamente duplicadas: 0
Loan_ID únicos: True


### 2.5 Estadísticas de las variables numéricas


In [9]:
df[["Loan_Amount", "Term_Days", "Monthly_Income", "Age",
    "Historical_Loan_Count", "Internal_Score", "Total_Due",
    "Current_Loan_Novations", "Days_Since_First_Loan",
    "Is_1Plus_DPD"]].describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
Loan_Amount,10000.0,370724.0,205279.3,100000.0,230000.0,300000.0,500000.0,1.000000e+06
Term_Days,10000.0,29.0,4.1,5.0,30.0,30.0,30.0,3.000000e+01
Monthly_Income,10000.0,3207502.1,18068624.7,110000.0,1800000.0,2500000.0,3600000.0,1.800000e+09
Age,10000.0,35.9,11.5,18.0,27.0,34.0,43.0,8.800000e+01
Historical_Loan_Count,10000.0,5.4,7.0,0.0,1.0,3.0,7.0,8.200000e+01
Internal_Score,1837.0,591.0,114.9,100.0,491.0,595.0,685.0,9.390000e+02
Total_Due,10000.0,464153.0,231340.6,114410.0,297558.0,379957.0,580277.0,1.251371e+06
Current_Loan_Novations,10000.0,1.3,1.9,0.0,0.0,0.0,2.0,8.000000e+00
Days_Since_First_Loan,8787.0,786.5,825.3,-224.0,86.0,566.0,1281.5,4.187000e+03
Is_1Plus_DPD,10000.0,0.6,0.5,0.0,0.0,1.0,1.0,1.000000e+00


- `Loan_Amount` va de 100 mil a 1 millón.
- `Monthly_Income` un dato llega a 1.800 millones, muy lejos del resto.
- `Age` va de 18 a 88 años.
- `Historical_Loan_Count` llega hasta 82 créditos.
- `Days_Since_First_Loan` tiene valores negativos.

Se dejan fuera Loan_ID, Client_ID y City_Id por ser identificadores.

### 2.6 Variables categóricas
como se concentran dichas variables

In [10]:
for col in ["Tipo_Cliente", "Employment_Type", "Dispersion_Method", "Loan_Purpose", "Estado_Credito", "Rango_NPL"]:
    print(col)
    print((df[col].value_counts(normalize=True) * 100).round(1).to_string())
    print()

Tipo_Cliente
Tipo_Cliente
Recurrente    79.4
Nuevo         20.6

Employment_Type
Employment_Type
Empleado         57.0
Otro             23.4
Independiente    17.6
Pensionado        2.0

Dispersion_Method
Dispersion_Method
Cuenta_Bancaria    99.2
Billetera           0.8
Efecty              0.0

Loan_Purpose
Loan_Purpose
Otros       99.9
Negocio      0.1
Arriendo     0.0

Estado_Credito
Estado_Credito
PAGADO                   71.3
MORA                     19.1
DESEMBOLSADO              9.6
PENDIENTE DE NOVACIÓN     0.1

Rango_NPL
Rango_NPL
03. Untimely Payment (1+ DPD)    48.6
02. Timely Payment               41.8
01. Current (Not Due)             9.6



`Loan_Purpose` es "Otros" en casi todos los casos y `Dispersion_Method` es cuenta bancaria en casi todos.

### 2.7 Variable objetivo (target)


In [11]:
print(df["Is_1Plus_DPD"].value_counts())
print(f"\nTasa de mora: {df['Is_1Plus_DPD'].mean():.1%}")

Is_1Plus_DPD
1    5819
0    4181
Name: count, dtype: int64

Tasa de mora: 58.2%


5.819 de 10.000 créditos están en mora (58,2%). Para microcrédito es una cifra muy alta

---
# Sección 3. Hallazgos de la base

## 3.1 Columnas del diccionario que no llegaron (PDF a Base de datos)

In [12]:
diccionario = {
    "Clients":     ["Client_ID", "Tipo_Cliente", "Monthly_Income", "Age", "City_Id", "Region",
                    "Employment_Type", "Internal_Score", "Rapiplazo_Score"],
    "Loans":       ["Loan_ID", "Client_ID", "Fecha_Desembolso", "Fecha_Vencimiento", "Loan_Amount",
                    "Term_Days", "Dispersion_Method", "Loan_Purpose", "Historical_Loan_Count",
                    "Estado_Credito_Anterior"],
    "Performance": ["Loan_ID", "Total_Due", "Current_Loan_Novations", "Dias_posterior_ejecucion",
                    "Rango_Estado_Anterior", "Days_Since_First_Loan", "Is_1Plus_DPD"],
}
reales = {"Clients": clients.columns, "Loans": loans.columns, "Performance": perf.columns}

no_llegaron, no_documentadas = [], []
for tabla, columnas in diccionario.items():
    no_llegaron    += [{"Tabla": tabla, "Columna": c} for c in columnas if c not in reales[tabla]]
    no_documentadas += [{"Tabla": tabla, "Columna": c} for c in reales[tabla] if c not in columnas]

print("En el diccionario pero NO la base de datos:")
display(pd.DataFrame(no_llegaron))
print("En la base de datos pero NO en el diccionario:")
display(pd.DataFrame(no_documentadas))

En el diccionario pero NO la base de datos:


,Tabla,Columna
0,Clients,Rapiplazo_Score
1,Loans,Estado_Credito_Anterior
2,Performance,Dias_posterior_ejecucion
3,Performance,Rango_Estado_Anterior


En la base de datos pero NO en el diccionario:


,Tabla,Columna
0,Loans,Estado_Credito
1,Performance,Rango_NPL


Dos columnas del diccionario no llegaron en los archivos: `Rapiplazo_Score` y `Dias_posterior_ejecucion`.

`Dias_posterior_ejecucion` no se puede crear: haría falta la fecha real de pago, que no está en ninguna de las tres bases.

- No se inventan las columnas faltantes.
- Sin `Dias_posterior_ejecucion` no hay curvas de mora a 7, 15 y 30 días. Se presenta como método propuesto y limitación.
- Sin `Rapiplazo_Score`, el único puntaje es `Internal_Score`.

## 3.2 `Rango_NPL` y `Estado_Credito` reproduce el target (tabla cruzada de variables)

In [13]:
print("Rango_NPL contra el target:")
display(pd.crosstab(df["Rango_NPL"], df["Is_1Plus_DPD"], margins=True))

print("Estado_Credito contra el target:")
display(pd.crosstab(df["Estado_Credito"], df["Is_1Plus_DPD"], margins=True))

Rango_NPL contra el target:


Is_1Plus_DPD,0,1,All
Rango_NPL,,,
01. Current (Not Due),0,956,956
02. Timely Payment,4181,0,4181
03. Untimely Payment (1+ DPD),0,4863,4863
All,4181,5819,10000


Estado_Credito contra el target:


Is_1Plus_DPD,0,1,All
Estado_Credito,,,
DESEMBOLSADO,4,954,958
MORA,9,1897,1906
PAGADO,4167,2963,7130
PENDIENTE DE NOVACIÓN,1,5,6
All,4181,5819,10000


- `Rango_NPL` coincide con el target sin excepciones: los 4.181 "Timely Payment" son todos 0, y los 4.863 "Untimely Payment" y los 956 "Current (Not Due)" son todos 1.
- `Estado_Credito` se comporta parecido: "MORA" casi siempre es 1.
- Ambas variables describen lo que pasó con el crédito después del desembolso.
`Rango_NPL` es **fuga de información**: informacion que no existe al momento de aprobar, y un modelo que la use daría un resultado perfecto (falso) que no se repetiría en producción. Queda fuera del modelo, igual que `Estado_Credito`, que solo se usa para medir pérdidas.

## 3.3 (956) créditos no vencidos marcados como mora

In [14]:
ES_NO_VENCIDO = df["Rango_NPL"].str.startswith("01")
ES_MOROSO     = df["Rango_NPL"].str.startswith("03")

# Estimar el corte: el último vencimiento entre morosos y el primero entre no vencidos deben quedar pegados
ultimo_venc_moroso      = df.loc[ES_MOROSO, "Fecha_Vencimiento"].max()
primer_venc_no_vencido  = df.loc[ES_NO_VENCIDO, "Fecha_Vencimiento"].min()
print("Último vencimiento entre morosos:     ", ultimo_venc_moroso.date())
print("Primer vencimiento entre no vencidos: ", primer_venc_no_vencido.date())

corte = ultimo_venc_moroso          # fecha de corte estimada
print("\nFecha de corte estimada:", corte.date())

# Créditos cuyo vencimiento es posterior al corte
posteriores = df[df["Fecha_Vencimiento"] > corte]
print("\nCréditos con vencimiento posterior al corte:", len(posteriores))
display(pd.crosstab(posteriores["Rango_NPL"], posteriores["Is_1Plus_DPD"]))

# Población de trabajo: solo créditos que ya llegaron a su vencimiento
df_vencidos = df[df["Fecha_Vencimiento"] <= corte].copy()
print("Créditos que se conservan (df_vencidos):", len(df_vencidos))

Último vencimiento entre morosos:      2026-06-15
Primer vencimiento entre no vencidos:  2026-06-16

Fecha de corte estimada: 2026-06-15

Créditos con vencimiento posterior al corte: 1031


Is_1Plus_DPD,0,1
Rango_NPL,,
01. Current (Not Due),0,956
02. Timely Payment,75,0


Créditos que se conservan (df_vencidos): 8969


 Un crédito que todavía no llegó a su fecha de pago no puede tener 1 día de atraso. Verificando son 956 "Current (Not Due)" cumplen eso. Para ello hace falta saber la fecha de corte de la extracción, que no viene en la base, así que se estima con las fechas de vencimiento.

 **Análisis.**

- El último vencimiento entre los morosos es el 15 de junio de 2026 y el primero entre los "no vencidos" es el 16. Con eso se estima la **fecha de corte de la extracción en el 15 de junio de 2026**.
- Los 956 créditos "Current (Not Due)" vencen después de esa fecha, así que **todavía no pueden tener 1 día de atraso**, y aun así están todos marcados como mora.
- Con la misma regla (vencimiento posterior al corte) quedan también 75 créditos que ya habían pagado por adelantado y tienen target 0. En total son **1.031 créditos**.

**Conclusión.**

- Se excluyen los 1.031 créditos con vencimiento posterior al corte: todavía no se conoce su resultado.
- Quedan **8.969 créditos (`df_vencidos`)**, que son la población de trabajo desde aquí.
- La fecha de corte es una estimación hecha con las fechas de vencimiento.

## 3.4 Mora alta y sesgada en los últimos meses
La mora global es 58,2%. se oberva cómo cambia por mes de desembolso y qué pasa al quitar los créditos que aún no vencen.

In [15]:
mes_todos    = df["Fecha_Desembolso"].dt.to_period("M")
mes_vencidos = df_vencidos["Fecha_Desembolso"].dt.to_period("M")

por_mes = pd.DataFrame({
    "creditos_total":     df.groupby(mes_todos).size(),
    "mora_total_%":       (df.groupby(mes_todos)["Is_1Plus_DPD"].mean() * 100).round(1),
    "creditos_vencidos":  df_vencidos.groupby(mes_vencidos).size(),
    "mora_vencidos_%":    (df_vencidos.groupby(mes_vencidos)["Is_1Plus_DPD"].mean() * 100).round(1),
})
display(por_mes)

print(f"Mora base completa:    {df['Is_1Plus_DPD'].mean():.1%}  ({df['Is_1Plus_DPD'].sum():,} de {len(df):,})")
print(f"Mora solo vencidos:    {df_vencidos['Is_1Plus_DPD'].mean():.1%}  ({df_vencidos['Is_1Plus_DPD'].sum():,} de {len(df_vencidos):,})")

print("\nDe los morosos (vencidos), cuántos ya pagaron según Estado_Credito:")
display(df_vencidos.loc[df_vencidos["Is_1Plus_DPD"] == 1, "Estado_Credito"].value_counts().to_frame("creditos"))

,creditos_total,mora_total_%,creditos_vencidos,mora_vencidos_%
Fecha_Desembolso,,,,
2025-10,1291,53.9,1291,53.9
2025-11,1047,55.2,1047,55.2
2025-12,1222,55.1,1222,55.1
2026-01,1285,54.4,1285,54.4
2026-02,1083,56.0,1083,56.0
2026-03,1282,51.8,1282,51.8
2026-04,1240,56.6,1240,56.6
2026-05,989,68.3,514,47.7
2026-06,561,93.8,5,0.0


Mora base completa:    58.2%  (5,819 de 10,000)
Mora solo vencidos:    54.2%  (4,863 de 8,969)

De los morosos (vencidos), cuántos ya pagaron según Estado_Credito:


,creditos
Estado_Credito,
PAGADO,2963
MORA,1897
PENDIENTE DE NOVACIÓN,3


**Análisis.**
- De octubre a abril la mora se mantiene estable entre 52% y 57%.
- En mayo sube a 68% y en junio a 94%. Es un **efecto de los créditos aún no vencidos**: al quitarlos, mayo baja a 48% y junio queda con solo 5 créditos.
- La mora baja de **58,2% a 54,2%** al trabajar solo con créditos vencidos.
- Sigue siendo alta para microcrédito. Una parte se explica porque el target es muy estricto (1 día de atraso): de los 4.863 morosos, **2.963 ya están pagados** (61%) y **1.897 siguen en mora** (39%).

**Conclusión.**
- Se usa `df_vencidos` para analizar y modelar.
- La mora de 54% se deja como **observación**: mezcla atrasos de 1 día con mora profunda.
- Se propone un **segundo target de "mora grave"** (créditos en estado MORA, 1.897 de 8.969, el 21%) para estimar el costo real del punto de corte.

## 3.5 Novaciones de 6 o más: mora casi segura

`Current_Loan_Novations` cuenta las veces que *este* crédito fue extendido o renegociado. cómo cambia la mora según el número de novaciones y si existe un punto de quiebre (default).

In [16]:
tabla = df_vencidos.groupby("Current_Loan_Novations")["Is_1Plus_DPD"].agg(creditos="size", mora="mean")
tabla["mora_%"] = (tabla["mora"] * 100).round(1)
display(tabla[["creditos", "mora_%"]])

seis_o_mas = df_vencidos["Current_Loan_Novations"] >= 6
print("Con 6 o más novaciones:")
print("  créditos:", seis_o_mas.sum(), "| morosos:", df_vencidos.loc[seis_o_mas, "Is_1Plus_DPD"].sum(),
      f"| mora: {df_vencidos.loc[seis_o_mas, 'Is_1Plus_DPD'].mean():.1%}")
print("Con 5 o menos novaciones:")
print("  créditos:", (~seis_o_mas).sum(), f"| mora: {df_vencidos.loc[~seis_o_mas, 'Is_1Plus_DPD'].mean():.1%}")

# En la base completa: cuántos de los que tienen 6+ no tienen historial
alto = df[df["Current_Loan_Novations"] >= 6]
print("\nBase completa con 6+ novaciones:", len(alto), "| sin historial previo:", (alto["Historical_Loan_Count"] == 0).sum())

,creditos,mora_%
Current_Loan_Novations,,
0,4941,49.8
1,1437,57.3
2,751,51.5
3,532,52.8
4,361,49.6
5,443,51.9
6,224,100.0
7,235,99.6
8,45,100.0


Con 6 o más novaciones:
  créditos: 504 | morosos: 503 | mora: 99.8%
Con 5 o menos novaciones:
  créditos: 8465 | mora: 51.5%

Base completa con 6+ novaciones: 508 | sin historial previo: 244


- Con 0 a 5 novaciones la mora se mueve entre 49,6% y 57,3%, **sin una tendencia clara**.
- Con 6, 7 u 8 novaciones, **los créditos están en mora (99,8%)**. El salto es contundente, sin valores intermedios. Eso es típico de una regla administrativa (por ejemplo, "a la sexta novación el crédito pasa a mora").
- Es una variable **posterior al desembolso**: una novación solo ocurre después de entregar el dinero, normalmente porque el cliente no pudo pagar. Al aprobar, su valor real es 0.
- Esos créditos (5% de la base) el modelo los clasificaría perfectamente, lo que **inflaría AUC, Gini y KS** sin aportar nada real.

## 3.6 Ingreso con topes.


In [17]:
ingreso = df["Monthly_Income"]

def pesos(x):
    # Formato pesos: $1.800.000.000
    return "$" + f"{x:,.0f}".replace(",", ".")

resumen = ingreso.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])
for estadistica, valor in resumen.items():
    if estadistica == "count":
        print(f"{estadistica:>6}: {valor:,.0f}".replace(",", "."))
    else:
        print(f"{estadistica:>6}: {pesos(valor):>16}")

print("\nRegistros exactamente en", pesos(10_000_000), ":", (ingreso == 10_000_000).sum())
print("Registros sobre", pesos(10_000_000), ":          ", (ingreso > 10_000_000).sum(), "| máximo:", pesos(ingreso.max()))

 count: 10.000
  mean:       $3.207.502
   std:      $18.068.625
   min:         $110.000
    1%:       $1.000.000
   25%:       $1.800.000
   50%:       $2.500.000
   75%:       $3.600.000
   99%:      $10.000.000
   max:   $1.800.000.000

Registros exactamente en $10.000.000 : 184
Registros sobre $10.000.000 :           1 | máximo: $1.800.000.000


In [18]:
# Salario mínimo vigente según el año del desembolso (Decreto 1469 de 2025; 2026 se mantiene con el Decreto 0159 de 2026)
smmlv = df["Fecha_Desembolso"].dt.year.map({2025: 1_423_500, 2026: 1_750_905})
menor_smmlv = ingreso < smmlv
print(f"Ingreso menor a 1 SMMLV: {menor_smmlv.sum():,} ({menor_smmlv.mean():.1%})")

# Límites estadísticos: IQR sobre el logaritmo del ingreso
log_ing = np.log10(ingreso)
q1, q3 = log_ing.quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = 10 ** (q1 - 1.5 * iqr), 10 ** (q3 + 1.5 * iqr)
print(f"\nLímites por IQR (log10): {lim_inf:,.0f} a {lim_sup:,.0f}")
print("Por debajo del límite inferior:", (ingreso < lim_inf).sum(), "| por encima del superior:", (ingreso > lim_sup).sum())

Ingreso menor a 1 SMMLV: 1,901 (19.0%)

Límites por IQR (log10): 636,396 a 10,182,338
Por debajo del límite inferior: 11 | por encima del superior: 1


In [19]:
tramos = pd.cut(df_vencidos["Monthly_Income"], [0, 1e6, 1.5e6, 2e6, 3e6, 5e6, 1e10],
                labels=["hasta 1M", "1M a 1,5M", "1,5M a 2M", "2M a 3M", "3M a 5M", "más de 5M"])
mora_ingreso = df_vencidos.groupby(tramos, observed=True)["Is_1Plus_DPD"].agg(creditos="size", mora="mean")
mora_ingreso["mora_%"] = (mora_ingreso["mora"] * 100).round(1)
mora_ingreso[["creditos", "mora_%"]]

,creditos,mora_%
Monthly_Income,,
hasta 1M,163,77.9
"1M a 1,5M",1373,62.3
"1,5M a 2M",2164,54.1
2M a 3M,2444,51.6
3M a 5M,1833,52.5
más de 5M,992,49.1


- **Hay un tope en 10 millones:** 184 registros están exactamente en 10.000.000 y el percentil 99 también es 10.000.000.
- **Hay un error evidente:** un registro declara 1.800 millones, 180 veces el tope. Parece un error de digitación.
- Por estadística (IQR sobre el logaritmo), solo 12 registros quedan fuera de los límites: 11 por debajo de unos 636 mil y 1 por encima de unos 10,18 millones.
- **Ingresos bajos no son error:** 1.901 clientes (19%) declaran menos de un salario mínimo del periodo (1.423.500 en 2025 y 1.750.905 en 2026). Es compatible con trabajo informal.
- **El ingreso separa bien la mora:** 78% en quienes declaran hasta 1 millón y 49% en quienes declaran más de 5 millones.



## 3.7 `Internal_Score` con muchos nulos

In [20]:
n_nulos = df["Internal_Score"].isna().sum()
print(f"Nulos: {n_nulos:,} de {len(df):,} ({n_nulos / len(df):.1%})")

print("\n% de nulos por tipo de cliente:")
display((df.groupby("Tipo_Cliente")["Internal_Score"].apply(lambda s: s.isna().mean() * 100)).round(1).to_frame("% nulos"))

print("Mora según tenga o no score (solo vencidos):")
tiene_score = df_vencidos["Internal_Score"].notna().map({True: "con score", False: "sin score"})
res = df_vencidos.groupby(tiene_score)["Is_1Plus_DPD"].agg(creditos="size", mora="mean")
res["mora_%"] = (res["mora"] * 100).round(1)
display(res[["creditos", "mora_%"]])

print(df["Internal_Score"].describe().round(1).to_string())

Nulos: 8,163 de 10,000 (81.6%)

% de nulos por tipo de cliente:


,% nulos
Tipo_Cliente,
Nuevo,71.3
Recurrente,84.3


Mora según tenga o no score (solo vencidos):


,creditos,mora_%
Internal_Score,,
con score,1729,70.0
sin score,7240,50.5


count    1837.0
mean      591.0
std       114.9
min       100.0
25%       491.0
50%       595.0
75%       685.0
max       939.0


- Solo el **18%** de los créditos tiene score (1.837 de 10.000).
- Es raro que falte **más en los clientes Recurrentes (84%) que en los Nuevos (71%)**: un score interno e histórico debería existir sobre todo para quienes ya tienen historia con la empresa.

## 3.8 Solo 21% de la base tiene perfil RapiFlex

Se  describe RapiFlex como créditos de 100 mil a 200 mil pesos a 30 días. La base no trae una columna de producto, así que se crea.

In [21]:
df["es_rapiflex"] = df["Loan_Amount"].between(100_000, 200_000) & (df["Term_Days"] == 30)
df_vencidos = df[df["Fecha_Vencimiento"] <= corte].copy()   # se refresca para incluir la bandera

print(f"Créditos con perfil RapiFlex: {df['es_rapiflex'].sum():,} ({df['es_rapiflex'].mean():.1%})")
print("Créditos con plazo distinto de 30 días:", (df["Term_Days"] != 30).sum())

comparacion = df.groupby("es_rapiflex").agg(
    creditos=("Loan_ID", "size"),
    monto_mediano=("Loan_Amount", "median"),
    ingreso_mediano=("Monthly_Income", "median"),
    historial_mediano=("Historical_Loan_Count", "median"),
    pct_clientes_nuevos=("Tipo_Cliente", lambda s: (s == "Nuevo").mean() * 100),
).round(1)
display(comparacion)

mora_rf = df_vencidos.groupby("es_rapiflex")["Is_1Plus_DPD"].agg(creditos="size", mora="mean")
mora_rf["mora_%"] = (mora_rf["mora"] * 100).round(1)
display(mora_rf[["creditos", "mora_%"]])

Créditos con perfil RapiFlex: 2,063 (20.6%)
Créditos con plazo distinto de 30 días: 759


,creditos,monto_mediano,ingreso_mediano,historial_mediano,pct_clientes_nuevos
es_rapiflex,,,,,
False,7937,350000.0,2500000.0,4.0,14.2
True,2063,200000.0,1900000.0,1.0,45.4


,creditos,mora_%
es_rapiflex,,
False,7126,50.7
True,1843,67.7



- **2.063 créditos (20,6%)** cumplen el perfil RapiFlex. Hay 759 créditos con plazo distinto de 30 días.
- Es una población distinta: más clientes nuevos (45% contra 14%), menos historial (mediana de 1 contra 4) y menor ingreso (1,9 millones contra 2,5 millones).
- La mora también es mayor: **67,7% en RapiFlex contra 50,7% en el resto** (solo créditos vencidos).
- Se usa para **segmentar**: comparar el modelo general contra uno solo de RapiFlex y ver si las variables se comportan igual dentro y fuera de ese producto.

## 3.9 Observación: variables casi constantes

In [22]:
for col in ["Loan_Purpose", "Dispersion_Method"]:
    print(col)
    print((df[col].value_counts(normalize=True) * 100).round(2).to_string())
    print()

Loan_Purpose
Loan_Purpose
Otros       99.91
Negocio      0.07
Arriendo     0.02

Dispersion_Method
Dispersion_Method
Cuenta_Bancaria    99.19
Billetera           0.79
Efecty              0.02



- `Loan_Purpose` es "Otros" en el 99,9% de los créditos.
- `Dispersion_Method` es cuenta bancaria en el 99,2% (billetera 0,8% y Efecty 0,02%).
- Ninguna aporta señal, por lo que **se descartan como predictoras**.

3.10 Mora por otras variable


In [23]:
def mora_por(serie, nombre):
    r = df_vencidos.groupby(serie, observed=True)["Is_1Plus_DPD"].agg(creditos="size", mora="mean")
    r["mora_%"] = (r["mora"] * 100).round(1)
    print(nombre); display(r[["creditos", "mora_%"]])

mora_por(pd.cut(df_vencidos["Age"], [17, 21, 25, 35, 45, 55, 65, 100],
                labels=["18-21", "22-25", "26-35", "36-45", "46-55", "56-65", "66 o más"]), "Mora por banda de edad")
mora_por(pd.cut(df_vencidos["Historical_Loan_Count"], [-1, 0, 2, 5, 10, 20, 100],
                labels=["0", "1-2", "3-5", "6-10", "11-20", "más de 20"]), "Mora por historial de créditos")
mora_por(df_vencidos["Employment_Type"], "Mora por tipo de empleo")

Mora por banda de edad


,creditos,mora_%
Age,,
18-21,626,58.8
22-25,1169,56.5
26-35,3119,55.7
36-45,2363,53.1
46-55,1096,51.2
56-65,443,47.4
66 o más,153,47.7


Mora por historial de créditos


,creditos,mora_%
Historical_Loan_Count,,
0,1945,70.8
1-2,2113,56.0
3-5,1895,50.7
6-10,1663,48.1
11-20,1020,41.1
más de 20,333,36.6


Mora por tipo de empleo


,creditos,mora_%
Employment_Type,,
Empleado,5080,55.8
Independiente,1565,64.7
Otro,2142,43.3
Pensionado,182,47.8


---
# Sección 4. Preparación de los datos



## 4.1 Criterio técnico para decidir qué variables entran

 Una variable puede predecir muy bien y aun así no servir. Lo que importa es si existe **antes de aprobar el crédito**. Para decidir cada variable se aplican cuatro pruebas Y Post-colocación es todo lo que se genera después del desembolso: pagos, atrasos, estados, rangos de mora y novaciones.


**Por qué `Rango_NPL` y `Estado_Credito` no entran aunque parezcan importantes:** parecen importantes porque **son el target con otro nombre**. Importancia se mide en poder predictivo, pero validez se mide en disponibilidad.

**Novaciones.** No se modelan. La regla "6 o más novaciones equivale a mora" no puede ser una regla de originación, porque al aprobar un crédito sus novaciones son cero. Es una regla de **gestión de cartera y cobranza**.

**`Total_Due`.** Se incluye porque el total a pagar (capital, intereses y cargos) se conoce al desembolsar, y no hay evidencia en los datos de que incluya recargos por mora: los créditos aún no vencidos, que no pueden tener recargos, muestran la misma tasa mediana que los morosos.  Por eso se usa la **tasa implícita** (`Total_Due / Loan_Amount - 1`) y no el valor bruto (que casi repite el monto y el plazo), y se reporta un modelo **sin** ella para medir cuánto cambia el resultado.

## 4.2 Población de trabajo y partición entrenamiento y prueba

Hay que decidir con qué créditos se entrena el modelo y con cuáles se prueba.

**Partición: Se trabaja con los créditos desembolsados de octubre de 2025 a abril de 2026 que ya llegaron a su vencimiento. Mayo y junio se excluyen porque los créditos recientes todavía no tienen resultado. La base se divide al azar en 80% para entrenar y 20% para probar, manteniendo la misma proporción de mora en ambos grupos y con una semilla fija para que sea reproducible. Como cada cliente aparece una sola vez, no hay riesgo de que el mismo cliente quede en los dos grupos. Como control adicional, en la sección 6 se repite la evaluación entrenando con octubre a enero y probando con febrero a abril.**

In [24]:
from sklearn.model_selection import train_test_split

# POBLACION: créditos vencidos desembolsados antes de mayo de 2026
df_mod = df_vencidos[df_vencidos["Fecha_Desembolso"] < "2026-05-01"].copy()

# Variable auxiliar solo para evaluar pérdidas (NO es variable del modelo): crédito que sigue en estado MORA
df_mod["mora_grave"] = (df_mod["Estado_Credito"] == "MORA").astype(int)

train, test = train_test_split(df_mod, test_size=0.2, stratify=df_mod["Is_1Plus_DPD"], random_state=42)

print(f"Población de trabajo: {len(df_mod):,} créditos | mora: {df_mod['Is_1Plus_DPD'].mean():.1%}")
print(f"Entrenamiento: {len(train):,} créditos | mora: {train['Is_1Plus_DPD'].mean():.1%}")
print(f"Prueba:        {len(test):,} créditos | mora: {test['Is_1Plus_DPD'].mean():.1%}")
print("Rango de desembolso:", df_mod["Fecha_Desembolso"].min().date(), "a", df_mod["Fecha_Desembolso"].max().date())

Población de trabajo: 8,450 créditos | mora: 54.7%
Entrenamiento: 6,760 créditos | mora: 54.6%
Prueba:        1,690 créditos | mora: 54.7%
Rango de desembolso: 2025-10-01 a 2026-04-30


las dos partes tienen la misma tasa de mora, así que la prueba es comparable. Limitación: al excluir mayo y junio se pierde el último mes y medio de desembolsos.

## 4.3 Construcción de las variables

**Calcular límites con el grupo de entrenamiento**. Se obtienen el rango normal del ingreso, el tope del historial (percentil 99) y las medianas de score y días. Se calculan solo con entrenamiento para que la prueba no influya en las reglas.

**Transformar las variables:**
- **Ingreso:** se acota a los límites y se pasa a logaritmo (suavizacion de la varibale). Se agregan dos indicadores: ingreso atípico y menor a un salario mínimo.
- **Edad:** se agrupa en bandas (18-21 es la referencia).
- **Historial:** se acota en 32 y se pasa a logaritmo, con un indicador de exceso.
- **Cliente y ocupación:** cliente nuevo sí o no, y una columna por tipo de empleo (Empleado es la referencia).
- **Monto y plazo:** monto en logaritmo y plazo de 30 días sí o no.
- **Crear tres bloques opcionales:** tasa implícita (precio), score (80% de vacios) y DIAS donde: dias_primer_credito	Los días, con los vacíos rellenados con la mediana (548) y dias_faltante	1 si el dato estaba vacío, 0 si tenía valor para que modelo no lo tome comoun dato real si no creado.
- **Organizar las columnas en cuatro listas:** BASE (Ingreso, edad, historial, tipo de cliente, ocupación, monto, plazo), PRECIO (tasa_implicita), SCORE (internal_score y score_faltante) y DIAS, para sumarlas de a una en la sección 5.

In [25]:
SMMLV = {2025: 1_423_500, 2026: 1_750_905}
BANDAS_EDAD = ["18-21", "22-25", "26-35", "36-45", "46-55", "56-65", "66+"]

def calcular_limites(t):
    """Límites aprendidos solo con entrenamiento."""
    q1, q3 = np.log10(t["Monthly_Income"]).quantile([0.25, 0.75])
    return {"ing_inf": 10 ** (q1 - 1.5 * (q3 - q1)), "ing_sup": 10 ** (q3 + 1.5 * (q3 - q1)),
            "hist_tope": t["Historical_Loan_Count"].quantile(0.99),
            "score_med": t["Internal_Score"].median(), "dias_med": t["Days_Since_First_Loan"].median()}

def preparar(d, lim):
    ing, hist = d["Monthly_Income"], d["Historical_Loan_Count"]
    X = pd.DataFrame({
        "ingreso_log":         np.log10(ing.clip(lim["ing_inf"], lim["ing_sup"])),
        "ingreso_atipico":     ((ing < lim["ing_inf"]) | (ing > lim["ing_sup"])).astype(int),
        "ingreso_menor_smmlv": (ing < d["Fecha_Desembolso"].dt.year.map(SMMLV)).astype(int),
        "historial_log":       np.log1p(hist.clip(upper=lim["hist_tope"])),
        "historial_exceso":    (hist > lim["hist_tope"]).astype(int),
        "cliente_nuevo":       (d["Tipo_Cliente"] == "Nuevo").astype(int),
        "monto_log":           np.log10(d["Loan_Amount"]),
        "plazo_30":            (d["Term_Days"] == 30).astype(int),
        "tasa_implicita":      d["Total_Due"] / d["Loan_Amount"] - 1,
        "internal_score":      d["Internal_Score"].fillna(lim["score_med"]),
        "score_faltante":      d["Internal_Score"].isna().astype(int),
        "dias_primer_credito": d["Days_Since_First_Loan"].fillna(lim["dias_med"]),
        "dias_faltante":       d["Days_Since_First_Loan"].isna().astype(int),
    })
    # Edad y ocupación: una columna 0/1 por categoría (se omite la de referencia)
    edad = pd.get_dummies(pd.cut(d["Age"], [17, 21, 25, 35, 45, 55, 65, 100], labels=BANDAS_EDAD),
                          prefix="edad").drop(columns="edad_18-21")
    empleo = pd.get_dummies(pd.Categorical(d["Employment_Type"], ["Empleado", "Independiente", "Otro", "Pensionado"]),
                            prefix="empleo").drop(columns="empleo_Empleado")
    empleo.index = d.index
    return pd.concat([X, edad.astype(int), empleo.astype(int)], axis=1)

BASE   = ["ingreso_log", "ingreso_atipico", "ingreso_menor_smmlv", "historial_log", "historial_exceso",
          "cliente_nuevo", "monto_log", "plazo_30"] + [f"edad_{b}" for b in BANDAS_EDAD[1:]] + \
         ["empleo_Independiente", "empleo_Otro", "empleo_Pensionado"]
PRECIO = ["tasa_implicita"]
SCORE  = ["internal_score", "score_faltante"]
DIAS   = ["dias_primer_credito", "dias_faltante"]

lim = calcular_limites(train)
print({k: round(float(v), 1) for k, v in lim.items()})
preparar(train, lim).head(3).round(2)

{'ing_inf': 636396.1, 'ing_sup': 10182337.6, 'hist_tope': 32.0, 'score_med': 586.5, 'dias_med': 548.0}


,ingreso_log,ingreso_atipico,ingreso_menor_smmlv,historial_log,historial_exceso,cliente_nuevo,monto_log,plazo_30,tasa_implicita,internal_score,score_faltante,dias_primer_credito,dias_faltante,edad_22-25,edad_26-35,edad_36-45,edad_46-55,edad_56-65,edad_66+,empleo_Independiente,empleo_Otro,empleo_Pensionado
2085,6.70,0,0,2.64,0,0,5.81,0,0.23,586.5,1,1295.0,0,0,1,0,0,0,0,0,0,0
9073,6.48,0,0,0.00,0,0,5.26,1,0.39,685.0,0,61.0,0,0,0,0,0,1,0,0,0,1
1257,6.32,0,0,1.39,0,0,5.53,1,0.32,586.5,1,723.0,0,0,0,1,0,0,0,1,0,0


---
# Sección 5. Modelado

Construir un **Modelo de riesgo** dado un crédito nuevo, estimar qué probabilidad tiene de caer en mora. Y decidir cuál versión del modelo usar.

**Funciones:**
- **ajustar:** entrenar el modelo con los créditos de entrenamiento y devuelve sus predicciones, tanto para entrenamiento como para prueba.
- **metricas:** mide qué tan bueno es el modelo (AUC, Gini y KS, que se explican abajo).

Es el modelo principal es una **regresion logistica.** La idea es asignarle a cada variable un peso (suma o resta riesgo) y combina todo en una probabilidad de **mora entre 0% y 100%.** Por ejemplo, más historial resta riesgo y ser cliente nuevo lo suma. Se eligió porque es rápida (la decisión debe ocurrir en segundos) acudiendo a la politica de Cero friccion e inmediatez y sus variables son explicables facilmente.

Se mide con:
**AUC** (qué tan seguido un moroso recibe más riesgo que un buen pagador),
**Gini** (el mismo dato, 2 × AUC - 1) y
**KS** (la mayor distancia entre buenos y malos), siempre con créditos de prueba que el modelo no vio.




In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve

def metricas(y, p):
    auc = roc_auc_score(y, p)
    fpr, tpr, _ = roc_curve(y, p)
    return auc, 2 * auc - 1, (tpr - fpr).max()      # AUC, Gini, KS

def ajustar(base_train, base_test, columnas, tipo="logistica"):
    lim_ = calcular_limites(base_train)               # límites aprendidos solo con entrenamiento
    Xtr, Xte = preparar(base_train, lim_)[columnas], preparar(base_test, lim_)[columnas]
    if tipo == "logistica":
        modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    else:
        modelo = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=42)
    modelo.fit(Xtr, base_train["Is_1Plus_DPD"])
    return modelo, modelo.predict_proba(Xtr)[:, 1], modelo.predict_proba(Xte)[:, 1]

# ===== 1. Comparación de cinco modelos =====
modelos = {
    "A. Base":                                  (BASE,                         "logistica"),
    "B. Base + precio (MODELO PRINCIPAL)":      (BASE + PRECIO,                "logistica"),
    "C. B + Internal_Score":                    (BASE + PRECIO + SCORE,        "logistica"),
    "D. B con árboles (modelo más complejo)":   (BASE + PRECIO,                "arboles"),
    "E. C + días primer crédito (DESCARTADO)":  (BASE + PRECIO + SCORE + DIAS, "logistica"),
}
filas = []
for nombre, (cols, tipo) in modelos.items():
    _, p_tr_, p_te_ = ajustar(train, test, cols, tipo)
    filas.append({"Modelo": nombre,
                  "AUC entrenamiento": roc_auc_score(train["Is_1Plus_DPD"], p_tr_),
                  "AUC prueba": roc_auc_score(test["Is_1Plus_DPD"], p_te_),
                  "Gini prueba": metricas(test["Is_1Plus_DPD"], p_te_)[1],
                  "KS prueba": metricas(test["Is_1Plus_DPD"], p_te_)[2]})
display(pd.DataFrame(filas).set_index("Modelo").round(3))

# ===== 2. Evidencia de por qué se descarta Days_Since_First_Loan =====
sin_dato = df_mod["Days_Since_First_Loan"].isna()
print("Mora cuando Days_Since_First_Loan NO tiene dato:", f"{df_mod.loc[sin_dato, 'Is_1Plus_DPD'].mean():.1%}", f"({sin_dato.sum():,} créditos)")
print("Mora cuando SÍ tiene dato:                      ", f"{df_mod.loc[~sin_dato, 'Is_1Plus_DPD'].mean():.1%}")

# ===== 3. Qué pesa en el modelo principal (y guardar el modelo para las secciones 6 a 8) =====
modelo_B, p_tr, p_te = ajustar(train, test, BASE + PRECIO)
columnas_B = BASE + PRECIO

efecto = pd.Series(modelo_B[-1].coef_[0], index=columnas_B)
efecto = efecto.reindex(efecto.abs().sort_values(ascending=False).index).head(8)
print("\nLas 8 variables con mayor efecto en el modelo principal:")
display(pd.DataFrame({"efecto": efecto.round(2), "sobre la mora": np.where(efecto > 0, "la sube", "la baja")}))

,AUC entrenamiento,AUC prueba,Gini prueba,KS prueba
Modelo,,,,
A. Base,0.646,0.634,0.267,0.217
B. Base + precio (MODELO PRINCIPAL),0.651,0.638,0.277,0.221
C. B + Internal_Score,0.655,0.652,0.303,0.234
D. B con árboles (modelo más complejo),0.725,0.665,0.331,0.259
E. C + días primer crédito (DESCARTADO),0.720,0.729,0.459,0.342


Mora cuando Days_Since_First_Loan NO tiene dato: 97.5% (1,023 créditos)
Mora cuando SÍ tiene dato:                       48.8%

Las 8 variables con mayor efecto en el modelo principal:


,efecto,sobre la mora
monto_log,-0.26,la baja
historial_log,-0.25,la baja
plazo_30,0.22,la sube
tasa_implicita,-0.21,la baja
empleo_Otro,-0.17,la baja
cliente_nuevo,0.11,la sube
empleo_Independiente,0.10,la sube
ingreso_menor_smmlv,0.09,la sube


**Análisis.**
- **Modelo principal (B):** Gini de 0,277 en la prueba. Es modesto, y es lo que dicen las variables disponibles al aprobar.

- **El precio y el score aportan poco:** de A a B el Gini sube 0,010 y de B a C, 0,026 es muy bajo.

- **Árboles (D):** suben el Gini a 0,331, pero se ajustan mucho más a los datos de entrenamiento (AUC de 0,725 contra 0,665 en la prueba). La logística apenas cambia (0,651 contra 0,638).

- **`Days_Since_First_Loan` (E):** el Gini salta a 0,459 y la causa es que los créditos sin ese dato tienen 97,5% de mora, contra 48,8% en los que sí lo tienen.

- **Qué pesa en el modelo:** más monto, más historial y la categoría de empleo "Otro" bajan la mora. El plazo de 30 días, ser cliente nuevo o independiente y ganar menos de un salario mínimo la suben.

Que un monto mayor reduzca la mora no quiere decir que prestar más reduzca el riesgo: los montos altos ya se le dieron a clientes probados (escalera de cupos). Parte de la señal de monto, plazo y tasa es la política actual, no el cliente.

**Conclusión.** Se usa la **regresión logística B** porque es simple, rápida y fácil de explicar y va en concordancia con la politica de cero friccion e inmediatez, y la ganancia de los árboles es moderada. Los árboles quedan como modelo propuesta o de monitoreo. `Days_Since_First_Loan` queda fuera, y `Internal_Score` solo como sensibilidad hasta saber cuándo se calcula.


---
# Sección 6. Resultados, punto de corte y segmentos

Esta sección responde tres preguntas: **¿el modelo sirve?, ¿dónde se corta la aprobación? y ¿funciona igual para todos los clientes?**

## 6.1 ¿El modelo sirve?

In [27]:
prueba = test.assign(prob=p_te)
y_te = prueba["Is_1Plus_DPD"].values
auc, gini, ks = metricas(y_te, p_te)

rng = np.random.default_rng(42)
g = [2 * roc_auc_score(y_te[i], p_te[i]) - 1 for i in (rng.integers(0, len(y_te), len(y_te)) for _ in range(500))]
print(f"AUC {auc:.3f} | Gini {gini:.3f} (IC 95%: {np.percentile(g, 2.5):.3f} a {np.percentile(g, 97.5):.3f}) | KS {ks:.3f}\n")

# Control en el tiempo: entrenar oct a ene, probar feb a abr
tr_t = df_mod[df_mod["Fecha_Desembolso"] < "2026-02-01"]
te_t = df_mod[df_mod["Fecha_Desembolso"] >= "2026-02-01"]
_, _, p_t = ajustar(tr_t, te_t, BASE + PRECIO)
print(f"Control en el tiempo (entrenar oct a ene, probar feb a abr): Gini {metricas(te_t['Is_1Plus_DPD'], p_t)[1]:.3f}\n")

# Cinco grupos de riesgo: 1 = menor riesgo, 5 = mayor riesgo
prueba["grupo"] = pd.qcut(prueba["prob"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
grupos = prueba.groupby("grupo", observed=True).agg(creditos=("Loan_ID", "size"),
                                                    mora_1DPD=("Is_1Plus_DPD", "mean"), perdida=("mora_grave", "mean"))
display((grupos[["mora_1DPD", "perdida"]] * 100).round(1).rename(columns={"mora_1DPD": "mora_1DPD_%", "perdida": "perdida_%"}).join(grupos["creditos"]))

AUC 0.638 | Gini 0.277 (IC 95%: 0.231 a 0.332) | KS 0.221

Control en el tiempo (entrenar oct a ene, probar feb a abr): Gini 0.252



,mora_1DPD_%,perdida_%,creditos
grupo,,,
1,35.2,4.7,338
2,48.2,11.2,338
3,60.1,18.9,338
4,60.4,22.8,338
5,69.5,47.0,338


Análisis. El Gini es 0,277 (intervalo de 0,231 a 0,332) y el control en el tiempo da 0,252, así que el resultado no depende de haber mezclado meses. La mora sube de 35% a 70% entre el grupo de menor y el de mayor riesgo, y la pérdida real (créditos que siguen en estado MORA) sube de 4,7% a 47%: el modelo ordena todavía mejor respecto a dicha perdida.

Conclusión. El modelo separa de forma moderada pero estable, y sirve sobre todo para identificar a quienes terminan en pérdida.

## 6.2 ¿Dónde cortar la aprobación?


El KS valida que el modelo separa, pero no fija el corte. El corte sale de comparar lo que se gana con lo que se pierde. Con el supuesto de (positivo = mora), **aprobar a un moroso es un falso negativo** y cuesta el capital prestado, y **rechazar a un buen cliente es un falso positivo** y cuesta su margen y su valor futuro. Los dos errores no cuestan lo mismo.

**Cómo se calcula con los datos:**
- **Ganancia de un crédito que paga:** `Total_Due - Loan_Amount`, más un valor futuro del cliente.
- **Pérdida:** si el crédito esta en MORA, se pierde el capital (con un porcentaje de recuperación).
- Se usa el **resultado real de cada crédito de prueba**, así que no hace falta suponer una tasa de pérdida.

observacion: Los datos no traen recuperaciones ni costos de cobranza, y solo hay créditos aprobados: el corte aplica a la población con la política actual.

In [28]:
VALOR_FUTURO = [0, 30_000, 60_000]   # editable: ganancia extra por cliente que paga (COP)
RECUPERACION = 0.0                    # editable: fracción del capital que se recupera de un crédito en MORA

def utilidad(d, vf):
    """Ganancia total (en millones) de un grupo de créditos: margen si pagan, pérdida del capital si quedan en MORA."""
    margen = d["Total_Due"] - d["Loan_Amount"]
    return float(np.where(d["mora_grave"] == 1, -d["Loan_Amount"] * (1 - RECUPERACION), margen + vf).sum() / 1e6)

# ----- 1. Curva de aprobación: aprobar a los de menor riesgo primero -----
orden = prueba.sort_values("prob")

def fila_aprobacion(pct):
    ap = orden.iloc[: round(len(orden) * pct / 100)]
    return {"aprobacion_%": pct, "mora_1DPD_%": ap["Is_1Plus_DPD"].mean() * 100, "perdida_%": ap["mora_grave"].mean() * 100,
            **{f"utilidad_VF{vf // 1000}k_MM": utilidad(ap, vf) for vf in VALOR_FUTURO}}

curva = pd.DataFrame([fila_aprobacion(p) for p in range(100, 49, -5)]).set_index("aprobacion_%").round(1)
display(curva)
utilidades = curva.filter(like="utilidad")
print("Aprobación con mayor utilidad:\n" + utilidades.idxmax().to_string())
print("\nUtilidad aprobando todo:\n" + utilidades.loc[100].to_string())

# ----- 2. Corte donde el KS es máximo -----
fpr, tpr, umbrales = roc_curve(y_te, p_te)
aprob_ks = (p_te <= umbrales[(tpr - fpr).argmax()]).mean()
ap_ks = orden.iloc[: round(len(orden) * aprob_ks)]
print(f"\nCorte en el KS máximo: aprueba {aprob_ks:.0%} | utilidad {[round(utilidad(ap_ks, vf), 1) for vf in VALOR_FUTURO]} MM")

# ----- 3. Punto de equilibrio por segmento -----
def equilibrio(d):
    margen, monto = (d["Total_Due"] - d["Loan_Amount"]).median(), d["Loan_Amount"].median()
    u = d["mora_grave"].mean()                      # pérdida observada
    return {"creditos": len(d), "perdida_observada_%": u * 100,
            **{f"tolerable_VF{vf // 1000}k_%": (margen + vf) / (margen + vf + monto * (1 - RECUPERACION)) * 100 for vf in VALOR_FUTURO},
            "valor_futuro_necesario": max(0, u / (1 - u) * monto * (1 - RECUPERACION) - margen)}

segmentos = {"Clientes nuevos (primera vez)": df_mod["Tipo_Cliente"] == "Nuevo",
             "Clientes recurrentes":          df_mod["Tipo_Cliente"] == "Recurrente",
             "RapiFlex":                      df_mod["es_rapiflex"]}
display(pd.DataFrame({n: equilibrio(df_mod[m]) for n, m in segmentos.items()}).T.round(1).astype({"creditos": int}))

,mora_1DPD_%,perdida_%,utilidad_VF0k_MM,utilidad_VF30k_MM,utilidad_VF60k_MM
aprobacion_%,,,,,
100,54.7,20.9,28.9,69.0,109.1
95,53.6,18.9,38.1,77.2,116.3
90,52.6,17.0,44.7,82.6,120.5
85,51.8,15.5,48.3,84.7,121.1
80,51.0,14.4,50.0,84.7,119.4
75,50.2,13.3,52.0,85.0,118.0
70,49.1,12.4,53.1,84.1,115.2
65,48.6,11.9,51.4,80.4,109.4
60,47.8,11.6,49.6,76.5,103.3


Aprobación con mayor utilidad:
utilidad_VF0k_MM     70
utilidad_VF30k_MM    75
utilidad_VF60k_MM    85

Utilidad aprobando todo:
utilidad_VF0k_MM      28.9
utilidad_VF30k_MM     69.0
utilidad_VF60k_MM    109.1

Corte en el KS máximo: aprueba 43% | utilidad [46.5, 66.3, 86.1] MM


,creditos,perdida_observada_%,tolerable_VF0k_%,tolerable_VF30k_%,tolerable_VF60k_%,valor_futuro_necesario
Clientes nuevos (primera vez),1849,46.6,27.3,34.5,40.3,99447.7
Clientes recurrentes,6601,13.3,19.2,24.5,29.2,0.0
RapiFlex,1775,38.3,26.9,34.1,40.1,50266.1


## 6.3 ¿Funciona igual en todos los segmentos?

Hasta aquí el modelo se evaluó con todos los créditos juntos. Esta parte responde una pregunta de negocio: **¿el modelo funciona igual de bien para todos los tipos de cliente, o falla en algún grupo?** Si fallara en uno, por ejemplo en los clientes nuevos, habría que tratarlo con otras reglas u otro modelo.

Qué créditos se usan. Solo los **1.690 créditos de prueba**, es decir, el 20% que el modelo **nunca vio** al entrenarse. Se hace así porque medir el modelo con los créditos de entrenamiento daría un resultado demasiado bueno porque ya los conoce. Los de prueba simulan lo que pasaría con solicitudes nuevas.

Qué grupos se comparan: Clientes **nuevos contra recurrentes**, y **RapiFlex contra el resto de créditos** (los de mayor monto). El segmento "no bancarizado" de la prueba no existe en los datos, y `Dispersion_Method` no sirve porque el 99,2% usa cuenta bancaria (hallazgo 3.9), así que se usa nuevo contra recurrente como aproximación.



In [29]:
def gini_ic(y, p):
    rng = np.random.default_rng(42); g = []
    for _ in range(500):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max(): g.append(2 * roc_auc_score(y[i], p[i]) - 1)
    return np.percentile(g, 2.5), np.percentile(g, 97.5)

filas = []
for col, valor, nombre in [("Tipo_Cliente", "Nuevo", "Cliente nuevo"), ("Tipo_Cliente", "Recurrente", "Cliente recurrente"),
                           ("es_rapiflex", True, "RapiFlex"), ("es_rapiflex", False, "Resto de créditos")]:
    m_ = (prueba[col] == valor).values
    lo, hi = gini_ic(y_te[m_], p_te[m_])
    filas.append({"Segmento": nombre, "creditos": m_.sum(), "mora_%": round(y_te[m_].mean() * 100, 1),
                  "Gini": round(2 * roc_auc_score(y_te[m_], p_te[m_]) - 1, 3), "IC 95%": f"{lo:.2f} a {hi:.2f}"})
display(pd.DataFrame(filas).set_index("Segmento"))

,creditos,mora_%,Gini,IC 95%
Segmento,,,,
Cliente nuevo,361,66.5,0.189,0.05 a 0.31
Cliente recurrente,1329,51.5,0.257,0.20 a 0.32
RapiFlex,347,67.1,0.208,0.08 a 0.34
Resto de créditos,1343,51.5,0.245,0.19 a 0.31


**Análisis.** se midio el Gini por grupo y sale más bajo en los clientes nuevos (0,19) y en RapiFlex (0,21) que en los recurrentes (0,26) y en el resto de créditos (0,25). Sin embargo, los intervalos de confianza se traslapan casi por completo. Con solo 361 clientes nuevos en la prueba, esa diferencia puede ser azar, así que no puedo afirmar que el modelo funcione peor ahí. Lo que sí cambia con claridad es el nivel de mora: 66,5% en los nuevos contra 51,5% en los recurrentes.

**Conclusión.** Con estos datos no hay evidencia para construir scorecards separados, pero sí para usar **cortes de aprobación distintos** por segmento, porque el riesgo de cada grupo es distinto. Si con más volumen se confirma que el Gini de los nuevos es realmente bajo, separaría el scorecard y le sumaría variables transaccionales y de comportamiento (telco, formulario web, huella del dispositivo).

# Sección 7. Escenarios extra

## 7.1 Cómo implementaría el motor

**Flujo en segundos:** solicitud, validaciones duras (edad, identidad, antifraude), cálculo del score, regla de corte y cupo, decisión y registro.

- **Servicio de scoring por API** con el modelo logístico: liviano, rápido y explicable. Las variables se calculan con las mismas reglas de la sección 4, guardando los límites aprendidos en el entrenamiento (percentiles y tope de ingreso, tope de historial).

- **Registro de cada decisión:** variables de entrada, score, versión del modelo y regla aplicada. Sirve para auditoría y para reentrenar.

- **Versionado del modelo (V#)**, para poder volver atrás si algo falla.

- **Plan de contingencia:** si el servicio de scoring cae, se aplica una regla conservadora (por ejemplo, monto mínimo para recurrentes con buen historial y revisión para nuevos) en lugar de aprobar a ciegas o rechazar a todos.

- **Separar fraude y riesgo de crédito.** El score de crédito no debe cargar con detectar suplantaciones: diferenciar la mora de de un formulario falso es fraude disfrazado de default. La verificación de identidad y el antifraude van antes del score, y deben equilibrar seguridad con fricción cero.

## 7.3 Procesos y áreas con las que interactuar y validar

| Área | Qué aporta o valida |
|---|---|
| **Riesgos** | Dueño del modelo y de la política. Define el presupuesto de pérdida y aprueba los cortes. |
| **Datos y analítica** | Calidad de los datos, construcción de variables, monitoreo del modelo. |
| **Tecnología** | Implementación del servicio, latencia, versionado, plan de contingencia. |
| **Producto** | Escalera de cupos, experiencia del cliente, impacto de la fricción en la conversión. |
| **Cobranza** | Definición de mora, retroalimentación sobre la calidad de los créditos, uso del modelo de cobranza (escenario A). |
| **Fraude e identidad** | Verificación de identidad y señales de fraude antes del score. |
| **Cumplimiento y jurídico** | Protección de datos personales (Ley 1581 de 2012) y habeas data financiero (Ley 1266 de 2008), uso de variables sensibles como la edad, trazabilidad y explicabilidad de las decisiones. A validar con el área jurídica. |
| **Finanzas** | Presupuesto de pérdida, rentabilidad por producto, valor futuro del cliente. |
| **Auditoría o validación independiente** | Revisión del modelo por alguien que no lo construyó. |

## 7.4 Cómo validaría la implementación y el proyecto


**Antes de salir:**
- **Validación independiente** del modelo: documentación, replicación de los resultados y revisión de información.
- **Prueba fuera de tiempo** (se encuentyra  en la seccion 6.1) y revisión de los datos de origen.
- Definir **antes** los criterios de éxito, por ejemplo: pérdida bajo el presupuesto con igual o mayor aprobación.

**Al salir:**
- Prueba en sombra: el motor calcula el score pero no decide. Se compara contra la política actual.
-  parte del tráfico se decide con el modelo nuevo.
- Piloto controlado: Como RapiFlex dura 30 días, el resultado llega rápido.

**Después de salir (monitoreo):**
- Cosechas: por semana o mes de desembolso, mirando la mora temprana a 7, 15 y 30 días. Para esto hace falta guardar la fecha real de pago.

- **PSI** mensual del score y de las variables, **Gini y KS** mensuales y **calibración** (mora predicha contra mora real).

- **Umbrales de alerta** definidos con anticipación (por ejemplo, PSI mayor a 0,25 o una caída sostenida del Gini) que disparan una recalibración.

## 7.6 Escenario A: un segundo caso de uso, el modelo de cobranza temprana

**Idea.** Las variables que se sacaron del modelo de originación (novaciones y estado del crédito) no se descartan: sirven para otro problema, **a quién llamar primero** una vez desembolsado el crédito.


---
# Sección 8. Conclusiones

## 8.1 Resultados principales

- **Modelo principal:** regresión logística con variables disponibles al momento de aprobar y la tasa implícita (bajo el supuesto de que `Total_Due` no incluye recargos por mora). En la prueba: AUC de 0,638, Gini de 0,277 (intervalo de 0,231 a 0,332) y KS de 0,221. Es un resultado aceptable.
- **Se sostiene en el tiempo:** entrenando con octubre a enero y probando con febrero a abril, el Gini es 0,252.
- **Ordena bien la pérdida real:** el grupo de menor riesgo tiene 4,7% de créditos en estado MORA y el de mayor riesgo 47,0%.
- **El corte sale de la utilidad, no del KS.** Aprobar entre 70% y 85% de las solicitudes maximiza la utilidad según el valor que se dé al cliente. Cortar en el KS máximo (43% de aprobación) pierde entre 12% y 29% de la utilidad.
- **El primer crédito pierde plata en promedio:** pérdida observada de 46,6% contra 27% a 40% tolerable. Los recurrentes sí son rentables (13,3%).
- **Un modelo de árboles rinde algo más** (Gini 0,331) pero se ajusta más a los datos de entrenamiento y es menos explicable. Queda como retador.


##8.2 Sugerencias para RappiCredit

- **Piloto de aprobación controlada.** Hoy solo se observan créditos aprobados, así que el modelo sirve para endurecer la política, no para flexibilizarla con evidencia de rehcazados. Se sugiere aprobar al azar entre 2% y 5% de las solicitudes que hoy se rechazan, con monto bajo de RapiFlex: la pérdida queda acotada y se obtiene la información que falta.

- **Mejor dato en origen:** Validar rangos  y guardar la **fecha real de pago**, que permitiría armar las cosechas. Es util y de bajo costo y mejora cualquier modelo futuro.

- **Confirmar:** la definición del target, la fecha de corte, el contenido de `Total_Due` y el momento de cálculo de `Internal_Score`.

- Reentrenar con los créditos de mayo y junio cuando maduren, y estimar un modelo de estas recuperaciones.

- Probar scorecards separados para clientes nuevos y pasar la logística a formato scorecard con tramos (WoE).